# sigma-rangeproof: a guided demo

This notebook runs the library end to end. It commits to a number, proves the number clears a threshold without revealing it, checks that the proof verifies, and then deliberately tries the cheats that are supposed to fail. By the end you should trust the thing because you watched it behave, not because the README said so.

If you have not read the docs, the one-line summary is: a Pedersen commitment hides a number, and a Sigma-protocol range proof shows the hidden number is at least some bound. Pure Python, no dependencies.

In [ ]:
# From a checkout of the repo:
#   pip install -e .
# or put ./src on the path. Then:
from sigma_rangeproof import commit, open_commit, prove_ge, verify_ge, RangeProof
print('imported')

## 1. Committing to a value

`commit` takes your number and returns two things: a commitment you can publish, and a blinding factor you keep secret. The commitment is a single large group element. On its own it tells an observer nothing about the number, because the random blinding smears it uniformly across the group.

The clearest way to see that hiding is real: commit to the same number twice and notice the commitments differ. Same value in, different group elements out, because each draw uses a fresh blinding.

In [ ]:
score = 740
commitment, blinding = commit(score)

print('commitment (first 32 hex chars):', hex(commitment)[:34], '...')
print('blinding   (first 32 hex chars):', hex(blinding)[:34], '...')

# Same value, second commitment -> different element. That is the hiding.
c2, _ = commit(score)
print('two commitments to the same value are equal:', commitment == c2)

# The holder can always re-open its own commitment to check its bookkeeping.
print('opens correctly to 740:', open_commit(commitment, 740, blinding))
print('opens to 741 (should be False):', open_commit(commitment, 741, blinding))

## 2. Proving a lower bound

Now the point of the library. We prove the committed value is at least 700, without revealing that it is 740.

`bits=10` says the value sits in a 10-bit window above the threshold, which covers `[700, 700+1024)`. Pick `bits` to match your value range; for a 0-1000 score, ten bits is right.

In [ ]:
proof = prove_ge(score, blinding, 700, bits=10)

print('proof carries', len(proof.commitments), 'bit-commitments and', len(proof.bit_proofs), 'bit-proofs')
print('verify >= 700:', verify_ge(commitment, 700, proof))

## 3. The proof only proves what it says

A proof is worth as much as the things it refuses to do. Four checks:

1. A proof for `>= 700` must not pass as a proof for `>= 720`. The verifier rebuilds its reference point from the threshold you give it, so a different threshold breaks the internal identity.
2. You cannot prove a value below the threshold. `prove_ge` raises rather than fake it.
3. Editing any number in the proof breaks it.
4. A proof does not transfer to a commitment it was not made for.

In [ ]:
# 1. Same proof, stricter threshold -> rejected.
print('verify >= 720 with a >=700 proof:', verify_ge(commitment, 720, proof))

# 2. Proving a false statement is impossible, not merely discouraged.
try:
    prove_ge(680, blinding, 700, bits=10)
except ValueError as e:
    print('proving 680 >= 700 raised ValueError:', e)

# 3. Tamper with one response and the bit proof fails.
from copy import deepcopy
bad = deepcopy(proof)
bad.bit_proofs[2]['z0'] += 1
print('verify tampered proof:', verify_ge(commitment, 700, bad))

# 4. A proof does not move to another commitment.
other, _ = commit(740)
print('verify against a different commitment:', verify_ge(other, 700, proof))

## 4. Sending a proof somewhere

Proofs and commitments are just numbers, so they travel as JSON. `to_dict` and `from_dict` round-trip a proof through hex strings. In a real system you would put the commitment somewhere public once, then ship proofs on demand.

In [ ]:
import json

# Holder serialises and sends.
envelope = json.dumps({'commitment': hex(commitment), 'proof': proof.to_dict()})
print('wire size:', len(envelope), 'bytes')

# Verifier receives, rebuilds, checks.
msg = json.loads(envelope)
rebuilt = RangeProof.from_dict(msg['proof'])
print('verify after a round trip:', verify_ge(int(msg['commitment'], 16), 700, rebuilt))

## 5. A worked example: gating on a private score

Here is the motivating use. Suppose every account has a reputation score between 0 and 1, kept private. A peer wants to accept mail only from accounts above 0.7, but you do not want to publish the raw scores. Commit to each score once; answer threshold questions with proofs.

Scores are real numbers, and the proof works on integers, so scale by 1000 and round. A score of 0.74 becomes 740, the threshold 0.7 becomes 700.

In [ ]:
def scale(x):
    return round(x * 1000)

def publish(score_float):
    c, r = commit(scale(score_float))
    return c, r  # publish c; the platform keeps r

def prove_above(score_float, blinding, threshold_float):
    return prove_ge(scale(score_float), blinding, scale(threshold_float), bits=10)

# A good-standing account.
alice_score = 0.74
alice_commit, alice_blind = publish(alice_score)
alice_proof = prove_above(alice_score, alice_blind, 0.70)
print('alice clears 0.70:', verify_ge(alice_commit, scale(0.70), alice_proof))

# A low-standing account cannot produce the proof at all.
bob_score = 0.55
bob_commit, bob_blind = publish(bob_score)
try:
    prove_above(bob_score, bob_blind, 0.70)
except ValueError:
    print('bob (0.55) cannot prove >= 0.70: no valid proof exists')

Notice what the verifier learned about Alice: that her score is at least 0.70. Not that it is 0.74. Not how it compares to anyone else. One bit of information, which is the whole design goal.

## 6. What it costs

Proving and verifying are each a handful of modular exponentiations per bit. On a natively built Python that is tens of milliseconds for `bits=10`. If the numbers below look terrible, check whether you are on an x86 Python under Rosetta on Apple silicon, which slows big-integer math by around a hundred times; build natively before judging.

In [ ]:
import time

t = time.time(); p = prove_ge(740, blinding, 700, bits=10); t_prove = time.time() - t
t = time.time(); ok = verify_ge(commitment, 700, p); t_verify = time.time() - t
print('prove : %.3f s' % t_prove)
print('verify: %.3f s  (ok=%s)' % (t_verify, ok))

## Where to go next

The `docs/` site walks through the math behind every step above: the group and the discrete-log assumption, why a Pedersen commitment hides and binds, the Schnorr and OR proofs underneath each bit, and how bit-decomposition turns into a range proof. The code in `src/sigma_rangeproof/` is short enough to read straight through once the math is in hand.